# driverisk: GPU run
Clones the repo, installs the pinned environment with uv, and runs a config.
**Status: not yet run on Colab/Kaggle.** First run = smoke test on stub data. Never upload raw video of people here.

In [ ]:
# 1. Check that a GPU is attached and record its type in the run log
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Clone. For a private repo use a read-only token typed at the prompt (do not save it).
import getpass, os
REPO = "MarufxAlchemist/Driver-risk-fusion"
BRANCH = "dev"
token = getpass.getpass("GitHub token (leave empty if the repo is public): ")
url = f"https://{token + '@' if token else ''}github.com/{REPO}.git"
if not os.path.isdir("/content/driver-risk-fusion"):
    !git clone --branch {BRANCH} {url} /content/driver-risk-fusion
%cd /content/driver-risk-fusion
!git log --oneline -1

In [ ]:
# 3. Install uv, then the exact environment (Python 3.11 + uv.lock)
!pip -q install uv
!uv sync --python 3.11

In [ ]:
# 4. The lock file holds the CPU build of PyTorch. Swap in a CUDA build for GPU runs.
# CHECK the index URL against https://pytorch.org/get-started/locally/ for the Colab driver.
CUDA_INDEX = "https://download.pytorch.org/whl/cu126"
!uv pip install --python .venv/bin/python --reinstall torch --index-url {CUDA_INDEX}
!uv run --no-sync python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

In [ ]:
# 5. Smoke test on synthetic data (about a minute). Must print 'stub pipeline OK'.
!uv run --no-sync python scripts/demo_stub.py

In [ ]:
# 6. Optional: unit tests (needs the dev group)
!uv sync --python 3.11 --group dev
!uv pip install --python .venv/bin/python --reinstall torch --index-url {CUDA_INDEX}
!uv run --no-sync pytest -q -m 'not slow'

In [ ]:
# 7. WAITING: scripts/train.py arrives with task P2-C3. Then, for example:
# !uv run --no-sync python scripts/train.py --config configs/<run>.yaml --seed 0
# Copy experiments/<run_id>/ (small files only) back to the repo; never commit checkpoints.
print('train.py not available yet')